# Ablasyon — her değişiklik tek başına (Colab Pro)

Kurallar `docs/protokol_ablasyon.md` dosyasındadır.

**GPU:** Aşama 2 T4 üzerinde koştu. "Hepsi birlikte" karşılaştırması adil olsun diye burada da **T4** seç. Daha hızlı bir GPU (L4/A100) seçersen raporda belirt.

Her model için 4 koşu (hepsi `--amp`), toplam 16 koşu:

| Koşu | Değişen |
|---|---|
| temel | hiçbiri: 224 piksel, hafif artırma, normal örnekleme |
| 384 | yalnız çözünürlük 384 |
| guclu | yalnız güçlü artırma |
| denge | yalnız sınıf dengeli örnekleme |

TTA her koşuda ayrıca ölçülür. "Hepsi birlikte" koşusu Aşama 2'den hazırdır.

Her koşunun sonuçları Google Drive'da `MyDrive/acne_ablasyon/` klasörüne zip olarak kaydedilir. Oturum koparsa notebook'u baştan çalıştır: Drive'daki zip'ler geri yüklenir ve biten koşular atlanır.

In [ ]:
MODELLER = ['resnet50', 'efficientnet_b0', 'convnext_tiny', 'densenet121']
DRIVE_KLASOR = '/content/drive/MyDrive/acne_ablasyon'

In [ ]:
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU kapalı. Çalışma zamanı türünü değiştirip yeniden başlat.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
Path(DRIVE_KLASOR).mkdir(parents=True, exist_ok=True)
print('Drive klasörü:', DRIVE_KLASOR)

In [ ]:
%cd /content
!rm -rf Final-project
!git clone https://github.com/bernademirer/Final-project.git
%cd /content/Final-project
!git log --oneline -3
assert 'guclu' in Path('src/train.py').read_text(encoding='utf-8'), "GitHub'daki kod eski."

In [ ]:
!pip -q install -r requirements.txt datasets

In [ ]:
import pandas as pd
from datasets import load_dataset

splits = pd.read_csv('data/splits.csv')
assert len(splits) == 1457
assert (splits.kume == 'test').sum() == 292
assert splits.file_name.nunique() == 1457

out = Path('data/images')
out.mkdir(parents=True, exist_ok=True)
need = set(splits.file_name) - {p.name for p in out.glob('*.jpg')}
if need:
    for row in load_dataset('Layered-Labs/ACNE04')['train']:
        if row['file_name'] in need:
            row['image'].save(out / row['file_name'])
print('jpg', len(list(out.glob('*.jpg'))))
assert len(list(out.glob('*.jpg'))) == 1457

In [ ]:
import zipfile
onceki = sorted(Path(DRIVE_KLASOR).glob('abl_*.zip'))
for z in onceki:
    zipfile.ZipFile(z).extractall('/content/Final-project')
print('Drive\'dan geri yüklenen koşu:', len(onceki))
for z in onceki:
    print(' ', z.name)

## Yardımcı fonksiyon

In [ ]:
import json
import os
import re
import shutil
import subprocess

VARYANTLAR = {
    'temel': '--aug on',
    '384': '--aug on --image-size 384',
    'guclu': '--aug guclu',
    'denge': '--aug on --balanced',
}


def calistir(cmd):
    print('$', cmd, flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    if p.wait():
        raise RuntimeError('komut hata verdi: ' + cmd)
    return ''.join(lines)


def olcum(cikti):
    d = {k: float(v) for k, v in re.findall(r'^(dogruluk|macro_f1|recall_\d) ([0-9.]+)', cikti, re.M)}
    return {'dogruluk': d['dogruluk'], 'macro_f1': d['macro_f1'], 'recall': [d[f'recall_{i}'] for i in range(4)]}


def kos(model, varyant):
    ad = f'{model}_abl_{varyant}'
    sonuc = Path(f'results/{ad}_eval.json')
    if sonuc.exists():
        print(ad, 'zaten tamam, atlandı')
        return
    agirlik = f'results/{ad}.pt'
    if not Path(agirlik).exists():
        calistir(f'python src/train.py --model {model} {VARYANTLAR[varyant]} --amp --name {ad}')

    ev = f'python src/evaluate.py --model {model} --weights {agirlik} --name {ad}'
    val = olcum(calistir(ev + ' --split val'))
    val_tta = olcum(calistir(ev + ' --split val --tta'))
    test = olcum(calistir(ev + ' --split test'))
    test_tta = olcum(calistir(ev + ' --split test --tta'))
    egitim = json.loads(Path(f'results/{ad}_val.json').read_text(encoding='utf-8'))
    sonuc.write_text(json.dumps({
        'model': model,
        'varyant': varyant,
        'val_macro_f1': val['macro_f1'],
        'val_macro_f1_tta': val_tta['macro_f1'],
        'test': test,
        'test_tta': test_tta,
        'epoch': egitim['epoch'],
        'gpu': torch.cuda.get_device_name(0),
    }, indent=2), encoding='utf-8')

    paket = [str(sonuc), f'results/{ad}_val.json', f'results/{ad}_egri.png',
             f'results/cm_{ad}.png', f'results/cm_{ad}_tta.png']
    zip_yolu = f'/content/abl_{model}_{varyant}.zip'
    with zipfile.ZipFile(zip_yolu, 'w') as z:
        for p in paket:
            if Path(p).exists():
                z.write(p)
    shutil.copy(zip_yolu, DRIVE_KLASOR)
    print(f'{ad}: val {val["macro_f1"]:.4f} (TTA {val_tta["macro_f1"]:.4f}) | test {test["macro_f1"]:.4f} (TTA {test_tta["macro_f1"]:.4f}) -> Drive')

## Tüm koşular

16 koşu sırayla çalışır; T4'te toplam yaklaşık 5–7 saat. Biten koşular atlanır, bu hücreyi istediğin kadar yeniden çalıştırabilirsin.

In [ ]:
for model in MODELLER:
    for varyant in VARYANTLAR:
        kos(model, varyant)
print('Tüm koşular tamam.')

## Özet

`src/ablasyon_ozet.py` GitHub'da varsa tablo ve figür burada üretilir ve Drive'a kopyalanır. Yoksa Drive'daki zip'leri bilgisayara indirip orada çalıştır.

In [ ]:
if Path('src/ablasyon_ozet.py').exists():
    calistir('python src/ablasyon_ozet.py')
    for ad in ('ablasyon_ozet.csv', 'sekil_ablasyon.png'):
        if Path(f'results/{ad}').exists():
            shutil.copy(f'results/{ad}', DRIVE_KLASOR)
    from IPython.display import Image, display
    display(Image('results/sekil_ablasyon.png'))
else:
    print('src/ablasyon_ozet.py GitHub\'da yok; özet bilgisayarda üretilecek.')